# Solar filament segmentation: train to submission

This notebook audits MAGFiLO, trains leakage-safe fold 0, predicts all test images, and decode-validates the final COCO RLE CSV. Enable a Kaggle GPU and attach both the competition data and this repository.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = 'https://github.com/TaiTranDang145/Solar-Filament-Segmentation.git'
PROJECT_ROOT = Path('/kaggle/working/Solar-Filament-Segmentation')
if (PROJECT_ROOT / '.git').exists():
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)

required = {'cv2': 'opencv-python-headless==4.12.0.88', 'pycocotools': 'pycocotools==2.0.10'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing])

sys.path.insert(0, str(PROJECT_ROOT))
for module_name in list(sys.modules):
    if module_name == 'solar_filament' or module_name.startswith('solar_filament.'):
        del sys.modules[module_name]

annotation_file = next(Path('/kaggle/input').rglob('MAGFiLO_1.0_Annotations_kaggle2026_train.json'))
DATA_ROOT = annotation_file.parents[1]
OUTPUT_ROOT = Path('/kaggle/working/artifacts')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Source code:', PROJECT_ROOT)
print('Competition data:', DATA_ROOT)
print('Output:', OUTPUT_ROOT)

In [ ]:
import torch

assert torch.cuda.is_available(), 'Enable GPU in Kaggle: Settings > Accelerator > GPU'
print('GPU:', torch.cuda.get_device_name(0))

## Audit the attached snapshot

A clean run reports 707 train files, 180 test files, and no errors.

In [ ]:
from solar_filament.data import audit_dataset

audit = audit_dataset(DATA_ROOT)
audit.as_dict()

## Train fold 0

The first run may download ImageNet ResNet50 weights. Set `pretrained_backbone=False` when internet is disabled. For a fast pipeline check, change `epochs` to 1; restore 20 before producing the final candidate.

In [ ]:
from solar_filament.training import TrainConfig, train

config = TrainConfig(
    data_root=str(DATA_ROOT),
    output_dir=str(OUTPUT_ROOT / 'baseline-fold-0'),
    fold=0,
    epochs=20,
    image_size=768,
    batch_size=2,
    num_workers=2,
    pretrained_backbone=True,
)
checkpoint = train(config)
checkpoint

In [ ]:
import json

metrics_path = OUTPUT_ROOT / 'baseline-fold-0' / 'metrics.jsonl'
metrics = [json.loads(line) for line in metrics_path.read_text().splitlines()]
best = max(metrics, key=lambda row: row['official_pq'])
print('Best checkpoint:', checkpoint)
print('Best metrics:', best)

## Infer and validate every RLE row

The run manifest records coverage, component areas, and latency for all 180 test images.

In [ ]:
from solar_filament.inference import infer_directory

submission_path = OUTPUT_ROOT / 'submission.csv'
report = infer_directory(
    checkpoint,
    DATA_ROOT / 'test' / 'test_images',
    submission_path,
)
report

In [ ]:
run = json.loads(submission_path.with_suffix('.run.json').read_text())
assert run['processed_images'] == 180
assert not report.errors
print(f"ready: {submission_path} ({report.rows} predicted instances)")